In [1]:
import torch
from ultralytics import YOLO
from spdconv import register_spdconv

# SART: YAML'daki "SPDConv" adini BU klasordeki spdconv.py'ye baglar.
# Cagrilmazsa ultralytics'e elle eklenmis inn_modules_v2.SPDConv devreye girer;
# o surumun state_dict anahtarlari farkli (conv.conv.weight vs conv.weight),
# yani SPD-Conv checkpoint'i SESSIZCE transfer olmaz (hata vermez, sadece
# transfer sayisi duser).
register_spdconv()

spdconv.SPDConv

In [2]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

Kullanilan cihaz: GPU (cuda:0)


In [3]:
# --- SPD-Conv + P2 ablasyonu ---
# Baseline'a gore IKI degisiklik var (bu ablasyonun tanimi bu):
#   1) butun stride-2 downsample'lar SPDConv (space-to-depth, bilgi kaybetmeyen)
#   2) P2/4 tespit dali eklendi -> Detect artik 4 seviyeli [P2,P3,P4,P5]
#
# P2 NEDEN: modules_v2.py'deki etiket istatistigine gore valid/test'in
# ~%38-39'u kucuk nesne (<32px). P2 (stride 4) tam bu dagilimi hedefliyor.
#
# AGIRLIK ZINCIRI - zayiftan iyiye, SON cagri kazanir:
#   baseline (duz yolo26n)   -> 355/902 tensor, %22.9 parametre
#   SPD-Conv (ablation_..v2) -> 360/902 tensor, %65.3 parametre
# Aradaki fark backbone'daki SPD katmanlarindan: SPDConv'un conv agirligi
# c2 x (4*c1) x k x k, duz Conv ise c2 x c1 x k x k. Duz baseline o
# katmanlarin sadece BatchNorm'unu doldurabiliyor, konvolusyonu bos birakiyor
# - ve bunlar modelin en agir katmanlari. Bu yuzden SPD-Conv checkpoint'i
# zincirin SONUNDA (kazanan konumda).
#
# Rastgele kalan ~542 tensor: P2 dali, 4. Detect basi ve genisleyen
# fuzyon dugumleri - hepsi bu mimariye YENI, beklenen durum.
model = (
    YOLO("spd-covn-p2.yaml")
    .load("../baseline/runs/detect/train/weights/best.pt")
    .load("../SPD-Conv/runs/detect/ablation_spd_conv_v2/weights/best.pt")
)

WARNING no model scale passed. Assuming scale='n'.
Transferred 356/902 items from pretrained weights
Transferred 360/902 items from pretrained weights


In [4]:
# --- SPD-Conv + P2 egitimi ---
# MALIYET: P2 dali stride-4 seviyesinde calisiyor, yani P3'un 4 kati
# aktivasyon alani. SPD-Conv da kanal sayisini gecici olarak 4x buyutuyor.
# Parametre 2.5M -> 4.55M. 8GB VRAM'de batch=16 sinirda olabilir;
# CUDA OOM alirsan batch=8 yap.
model.train(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    patience=100,
    plots=True,
    name="ablation_spd_conv_p2",
)

New https://pypi.org/project/ultralytics/8.4.137 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.12.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=spd-covn-p2.yaml, momentum=0.937, mosaic=1.0,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EA13492DE0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [5]:
import csv, glob
run = sorted(glob.glob("runs/detect/ablation_spd_conv_p2/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
print("\nEN IYI epoch %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]),
    float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])))

box1 = float(rows[0]["train/box_loss"])
print("\n[saglik] 1. epoch train/box_loss = %.2f -> %s" % (
    box1, "OK" if box1 < 20 else "!! KAYIP OLCEGI PATLAMIS, DURDUR"))
if len(rows) < 500:
    print("[uyari] kosu %d epoch'ta bitmis, 500 degil - yarida kesilmis olabilir" % len(rows))

runs/detect/ablation_spd_conv_p2/results.csv | epoch: 477
ep 1    box=3.17465  cls=5.42893  mAP50=0.0506  mAP50-95=0.01566
ep 2    box=2.17547  cls=2.71214  mAP50=0.29416 mAP50-95=0.12699
ep 475  box=0.75031  cls=0.38195  mAP50=0.88643 mAP50-95=0.63412
ep 476  box=0.73098  cls=0.37453  mAP50=0.88539 mAP50-95=0.63404
ep 477  box=0.72677  cls=0.35776  mAP50=0.8855  mAP50-95=0.6338

EN IYI epoch 377 -> P 0.907 R 0.823 mAP50 0.8885 mAP50-95 0.6427

[saglik] 1. epoch train/box_loss = 3.17 -> OK
[uyari] kosu 477 epoch'ta bitmis, 500 degil - yarida kesilmis olabilir


In [6]:
from pathlib import Path
best = Path(sorted(glob.glob("runs/detect/ablation_spd_conv_p2/weights/best.pt"))[-1])
print("test ediliyor:", best)

res = YOLO(str(best)).val(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    split="test",
    imgsz=640,
    device=device,
)
d = res.results_dict
print("\n=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)

test ediliyor: runs\detect\ablation_spd_conv_p2\weights\best.pt
Ultralytics 8.4.14  Python-3.12.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
spd-covn-p2 summary (fused): 160 layers, 4,435,216 parameters, 0 gradients
val: Fast image access  (ping: 0.00.0 ms, read: 152.357.7 MB/s, size: 26.7 KB)
val: Scanning C:\Users\ertug\Desktop\yolo\dataset2\yolo26\YOLO.v1i.yolo26\test\labels.cache... 279 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 279/279  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 325, len(boxes) = 1023. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 8.2it/s 2.2s0.1s
                   all        279       1023      0.893       0.83      0.886      0.649
                 crack        192